In [6]:
import json
import numpy as np
import pandas as pd
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
import os

In [7]:
load_dotenv(override=True)

True

In [8]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [9]:
model=ChatGroq(model="llama-3.3-70b-versatile", temperature=0.0)

In [10]:
SYMPTOMS = [
   "agitation", "angina_pectoris", "arm_weakness", "ascites", "asthenia", "calf_pain_walking", "chest_tightness", "chill", "confusion", "cough", "dark_urine", "diarrhea", "disorientation", "respiratory_distress", "dizziness", "dysarthria", "dyspnea", "dyspnea_on_exertion", "erythema", "facial_paresis", "fall", "fatigue", "feeling_hopeless", "fever", "frequent_urination", "haemorrhage", "auditory_hallucinations", "headache", "hemiplegia", "hyperkalemia", "hypokinesia", "hypotension", "hypoxemia", "irritable_mood", "joint_pain", "joint_swelling", "jugular_venous_distention", "leg_swelling", "lethargy", "loss_of_appetite", "low_energy", "malaise", "memory_loss", "mental_status_changes", "mood_depressed", "morning_stiffness", "nausea", "night_sweat", "non_productive_cough", "numbness", "oliguria", "orthopnea", "pain", "abdominal_pain", "chest_pain", "palpitation", "personality_changes", "pleuritic_pain", "chest_pressure", "productive_cough", "pruritus", "rale", "redness", "right_upper_quadrant_pain", "seizure", "shortness_of_breath", "sleeplessness", "snuffle", "social_withdrawal", "slurred_speech", "stiffness", "sweating", "swelling", "tachypnea", "sore_throat", "tremor", "unresponsiveness", "vomiting", "weepiness", "weight_gain", "wheezing", "worry", "yellow_eyes", "yellow_skin"
]

In [11]:
def extract_symptoms(patient_text: str) -> list:
    """
    Uses LLM to extract symptoms from patient text.
    Matches directly against the official symptom list.
    Returns a list of matched symptom strings from SYMPTOMS.
    """

    symptoms_formatted = SYMPTOMS.copy()

    system = SystemMessage(content=f"""You are a precise medical symptom extraction system.

        Your job is to read a patient's description and return ONLY the symptoms that are present, from the official symptom list provided below.

        OFFICIAL SYMPTOM LIST:
        {symptoms_formatted}

        STRICT RULES — follow every rule exactly, no exceptions:
        1. ONLY return symptoms from the official list above. Do not invent or add any symptom not in this list.
        2. Match symptoms by meaning, not just exact words. If the patient describes something that clearly means the same as a symptom in the list, include that symptom from the list.
        - Example: patient says "I feel dizzy" → match "dizziness"
        - Example: patient says "my heart is racing" → match "increased heart rate" and "palpitations"
        - Example: patient says "I can't sleep" → match "insomnia"
        - Example: patient says "tired all the time" → match "fatigue"
        3. NEGATION RULE — this is critical: if the patient says they do NOT have a symptom, or uses words like "no", "not", "without", "never", "doesn't", "don't", strictly exclude that symptom.
        - Example: "no fever" → do NOT include "fever"
        - Example: "I don't have a headache" → do NOT include "headache"
        - Example: "no chest pain but I feel dizzy" → include "dizziness", exclude "sharp chest pain"
        4. Only include symptoms that are clearly present or strongly implied. Do not guess.
        5. Return ONLY a valid JSON array of strings. No explanation, no markdown, no extra text.
        6. Each string in the array must exactly match a symptom from the official list word for word.

        Output format:
        ["symptom one", "symptom two", "symptom three"]""")

    human = HumanMessage(content=f"Patient description:\n{patient_text}")

    response = model.invoke([system, human])

    raw = response.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    matched_symptoms = json.loads(raw)

    # Safety filter — keep only symptoms that are actually in our list
    matched_symptoms = [s for s in matched_symptoms if s in SYMPTOMS]

    return matched_symptoms

In [12]:
def to_binary_vector(matched_symptoms: list) -> list:
    """
    Converts matched symptom list to a binary vector of size 377.
    Position of each symptom matches its index in SYMPTOMS list.
    """
    matched_set = set(matched_symptoms)
    return [1 if symptom in matched_set else 0 for symptom in SYMPTOMS]

In [13]:
def parse_symptoms(patient_text: str) -> dict:
    """
    Full pipeline: patient free text → binary vector of size 377.

    Returns:
        binary_vector    → list of 377 ints (0/1), feed into LightGBM
        matched_symptoms → list of matched symptom names from official list
    """
    matched = extract_symptoms(patient_text)
    binary_vector = to_binary_vector(matched)

    return {
        "binary_vector":    binary_vector,   # → directly into model.predict()
        "matched_symptoms": matched,          # → for display/debugging
    }

In [14]:
import pickle
# import lightgbm
with open(r"D:\ml_udemy\00 ML CODE\Projects\Digital Diagnosis v2\models\lightgbm_model.pkl", 'rb') as file:
    lgb=pickle.load(file)

with open(r"D:\ml_udemy\00 ML CODE\Projects\Digital Diagnosis v2\models\label_encoder.pkl", 'rb') as file:
    le=pickle.load(file)

In [15]:
def predict_top3(symptom_values: list):
    """Pass a list of 0/1 values matching your symptom columns."""
    x = np.array(symptom_values).reshape(1, -1)
    probs = lgb.predict_proba(x)[0]
    top3_idx = np.argsort(probs)[::-1][:3]

    print("Top 3 Predicted Diseases:")
    for rank, idx in enumerate(top3_idx, 1):
        disease = le.inverse_transform([idx])[0]
        prob = probs[idx] * 100
        print(f"  {rank}. {disease:20s} {prob:.2f}%")

In [16]:
test_input = "i am feeling anxiety and nervesnous, dizziness,"

result = parse_symptoms(test_input)

print("Matched Symptoms:")
for s in result["matched_symptoms"]:
    print(f"  ✓ {s}")

print(f"\nBinary vector size  : {len(result['binary_vector'])}")
print(f"Active symptoms     : {sum(result['binary_vector'])}")
print(f"Binary vector sample: {result['binary_vector']}")
print(f"top 3 predictions: {predict_top3(result['binary_vector'])}")

Failed to get info from https://api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info in LangSmith API. Please confirm your internet connection. ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /info (Caused by ConnectTimeoutError(<HTTPSConnection(host=\'api.smith.langchain.com\', port=443) at 0x1a192f1bbb0>, \'Connection to api.smith.langchain.com timed out. (connect timeout=10.0)\'))"))\nContent-Length: None\nAPI Key: lsv2_********************************************30')


Matched Symptoms:
  ✓ dizziness

Binary vector size  : 84
Active symptoms     : 1
Binary vector sample: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Top 3 Predicted Diseases:
  1. hyperlipidemia       29.71%
  2. hypercholesterolemia 14.87%
  3. hypertensive disease 9.53%
top 3 predictions: None


In [17]:
lgb_input=np.array(result['binary_vector']).reshape(1,-1)

In [18]:
le.inverse_transform(lgb.predict(lgb_input))

array(['hyperlipidemia'], dtype=object)